In [ ]:
from pathlib import Path
from typing import Literal
import pickle
import os, pathlib
while pathlib.Path.cwd().name != 'forecast_research':
    os.chdir('..') # Run from repo root
import tqdm
import tqdm.auto
tqdm.auto.tqdm = tqdm.tqdm
import pandas as pd

import python.utils as utils
import python.plots as plots
from python.error_metrics import MergedErrorMetrics, LoadedErrorMetrics, FilteredErrorMetrics, MetricsIndex as MI # Alias
from python.forecast_reconciliation import reconcile_calculate_serialize_error_metrics
from python.sampled_hierarchies import SampledHierarchies
from python.statistical_test import TrendType, l_test_stats, t_test_stats, TestResultsLatex
from python.time_series import BASE_FORECASTS, RECONCILED_FORECASTS
from python.aggregation import MONTHS, DAYS
from python.constants import AVAILABLE_LEVELS
from python.base_forecasting import MODEL_MIN_TRAIN_OBS

from itertools import product

In [ ]:
sh = SampledHierarchies(AVAILABLE_LEVELS, 3, 10)

datasets = ['web-traffic', 'M4', 'M5']
models = ['AutoETS', 'AutoARIMA', 'Prophet']

In [ ]:
daily_results = {}
monthly_results = {}
for d, m in tqdm.tqdm(product(datasets, models)):
    ID = f"{d}/{m}"
    tqdm.tqdm.write(ID)
    em = FilteredErrorMetrics(LoadedErrorMetrics(f"metrics/{ID}/sample_hiers_errors.csv"), MI(error_metric='nrmse'))
    monthly_results[ID] = sh.error_matrix(em, sh.index_matrix(granularity=MONTHS, forecast_type=RECONCILED_FORECASTS))
    daily_results[ID] = sh.error_matrix(em, sh.index_matrix(granularity=DAYS,   forecast_type=RECONCILED_FORECASTS))

with open("metrics/daily_error_matrices.pkl", 'wb') as f:
    pickle.dump(daily_results, f)

with open("metrics/monthly_error_results.pkl", 'wb') as f:
    pickle.dump(monthly_results, f)


In [ ]:
with open("metrics/daily_error_matrices.pkl", 'rb') as f:
    daily_results = pickle.load(f)

with open("metrics/monthly_error_results.pkl", 'rb') as f:
    monthly_results = pickle.load(f)

In [ ]:
test_fun = t_test_stats

In [ ]:
pd.DataFrame(
    {
        m: {
            d: test_fun(monthly_results[f"{d}/{m}"], 'winsorized', TrendType.DECREASING).p_value
            for d in datasets
        }
        for m in models
    }
)


In [ ]:
pd.DataFrame(
    {
        m: {
            d: test_fun(daily_results[f"{d}/{m}"], 'winsorized', TrendType.DECREASING).p_value
            for d in datasets
        }
        for m in models
    }
)

In [ ]:
model = 'AutoETS'
dataset = 'web-traffic'
stat = 'winsorized'
granularity = daily_results or monthly_results

plots.plot_sampled_error_matrix(sh, stat, stats_matrix=granularity[f"{dataset}/{model}"])